In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import root_mean_squared_log_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, LabelEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer

from xgboost import XGBRegressor

import random
# from skopt import BayesSearchCV
# from skopt.space import Real, Integer, Categorical

# First Step: Training Data Analysis

Let's take a closer look at the features in this dataset and determine if there are ways we can improve our analysis

In [2]:
dataset_df = pd.read_csv('./data/train.csv')

# Data Pre-processing

In [4]:
dataset_df_numeric_var = dataset_df.select_dtypes(include=[np.number]).drop(columns=['Id', 'SalePrice']).columns
dataset_df_categor_var = dataset_df.select_dtypes(include=[object]).columns
use_columns = dataset_df_numeric_var.append(dataset_df_categor_var)

In [5]:
X = dataset_df[use_columns]
y = dataset_df.SalePrice

In [6]:
# Define Data Pre-processing Pipeline
numerical_transform = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy="mean")),
    ('scaler', StandardScaler())
])
categorical_transform = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore')) # could try LabelEncoder instead
])
preprocessing = ColumnTransformer(transformers=[
    ('numerical', numerical_transform, dataset_df_numeric_var),
    ('categorical', categorical_transform, dataset_df_categor_var)
])

In [11]:
# Setup Model with Adaptive Training
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=58)

param_grid = {
    'n_estimators': range(50, 200, 50),
    'max_depth': range(5, 10),
    'learning_rate': [0.01, 0.1, 1.0]
}

training_pipeline = Pipeline(steps=[
    ('preprocess', preprocessing),
    ('model', GridSearchCV(XGBRegressor(),
                           param_grid,
                           cv=5,
                           scoring='neg_root_mean_squared_error'))
])

training_pipeline.fit(X_train, y_train)

,steps,"[('preprocess', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('numerical', ...), ('categorical', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [12]:
training_pipeline.named_steps['model'].best_params_

{'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 150}

In [13]:
y_pred = training_pipeline.predict(X_val)
root_mean_squared_log_error(y_val, y_pred)

0.14736603200435638

In [14]:
test_df = pd.read_csv('./data/test.csv')
X_test = test_df[use_columns]

y_pred = training_pipeline.predict(X_test)
proposed_submission = pd.DataFrame({'Id': test_df.Id, 'SalePrice': y_pred})
proposed_submission.to_csv('proposed_submission.csv', index=False)